# 04 – MobileNetV2 Transfer Learning

This notebook fine-tunes a **MobileNetV2** pre-trained on ImageNet for GTSRB traffic-sign classification.

MobileNetV2 is significantly lighter than ResNet-50, making it suitable for deployment on resource-constrained devices while still achieving strong accuracy.

**Strategy:**
1. Load MobileNetV2 with ImageNet weights.
2. Replace the final classifier with a 43-class head.
3. **Phase 1** – Train only the new head (backbone frozen).
4. **Phase 2** – Unfreeze the last few layers and fine-tune.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = Path('../data')
NUM_CLASSES = 43
BATCH_SIZE = 64
EPOCHS_HEAD = 5
EPOCHS_FINE = 10
LR_HEAD = 1e-3
LR_FINE = 5e-5
IMG_SIZE = 224

print(f'Using device: {DEVICE}')

## 1. Data loading

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.GTSRB(
    root=DATA_ROOT, split='train', download=True, transform=train_transform
)
test_dataset = datasets.GTSRB(
    root=DATA_ROOT, split='test', download=True, transform=val_transform
)

val_size   = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_dataset, val_dataset = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}  |  Test: {len(test_dataset):,}')

## 2. Model – MobileNetV2 with custom classifier

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)

# Replace final classifier
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable:,}')

## 3. Helper functions

In [ ]:
criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def run_training(model, train_loader, val_loader, optimizer, scheduler,
                 criterion, device, epochs, phase_label):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        if scheduler is not None:
            scheduler.step()
        elapsed = time.time() - t0
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        print(f'[{phase_label}] Epoch {epoch:2d}/{epochs} | '
              f'Train {tr_loss:.4f}/{tr_acc:.4f} | Val {va_loss:.4f}/{va_acc:.4f} | {elapsed:.1f}s')
    return history

## 4. Phase 1 – Train head only

In [ ]:
# Freeze backbone
for name, param in model.named_parameters():
    param.requires_grad = name.startswith('classifier')

optimizer_head = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD
)
history_head = run_training(
    model, train_loader, val_loader, optimizer_head, None,
    criterion, DEVICE, EPOCHS_HEAD, 'Phase-1 head'
)

## 5. Phase 2 – Fine-tune last feature blocks + classifier

In [ ]:
# Unfreeze the last 3 inverted residual blocks + classifier
unfreeze_from = len(list(model.features)) - 3
for i, (name, param) in enumerate(model.named_parameters()):
    block_idx = None
    parts = name.split('.')
    if parts[0] == 'features' and parts[1].isdigit():
        block_idx = int(parts[1])
    if block_idx is not None and block_idx >= unfreeze_from:
        param.requires_grad = True
    elif parts[0] == 'classifier':
        param.requires_grad = True

optimizer_fine = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINE
)
scheduler_fine = optim.lr_scheduler.CosineAnnealingLR(optimizer_fine, T_max=EPOCHS_FINE)

history_fine = run_training(
    model, train_loader, val_loader, optimizer_fine, scheduler_fine,
    criterion, DEVICE, EPOCHS_FINE, 'Phase-2 fine-tune'
)

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
    combined_train = history_head[f'train_{key}'] + history_fine[f'train_{key}']
    combined_val   = history_head[f'val_{key}']   + history_fine[f'val_{key}']
    ax.plot(combined_train, label='Train')
    ax.plot(combined_val,   label='Val')
    ax.axvline(EPOCHS_HEAD - 0.5, color='grey', linestyle='--', label='Phase 2 start')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(title); ax.legend()

plt.suptitle('MobileNetV2 – Training curves', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Test-set evaluation

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f'Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

## 8. Confusion matrix

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


y_pred, y_true = get_predictions(model, test_loader, DEVICE)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, ax=ax, cmap='Blues', xticklabels=range(43), yticklabels=range(43),
            linewidths=0.3, annot=False)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('MobileNetV2 – Confusion matrix (test set)')
plt.tight_layout()
plt.show()

## 9. Per-class accuracy report

In [ ]:
print(classification_report(y_true, y_pred, digits=3))

## 10. Save model weights

In [ ]:
torch.save(model.state_dict(), '../data/mobilenetv2_weights.pth')
print('Model weights saved to ../data/mobilenetv2_weights.pth')